<h1 style="color: #CEDDF4;">DEPI Round 4, MS Data Engineer and AI Track</h1>
<h2 style="color: #CEDDF4;" >Final Project: Gold and Oil Prediction System</h2>
<h3 style="color: #CEDDF4;" > Part (1): Python Code for Data Gathering and Data Cleaning</h3>

<h4 style="color: #CEDDF4;" >1. Import Libraries</h4>

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import requests
import pandas as pd
import time
import csv
from itertools import zip_longest
import math
from pathlib import Path
import os
import yfinance as yf
from datetime import datetime
import io
import re

In [2]:
base_dir = Path.cwd()
#raw data path to contain all raw files gathered either manually, web scraping or APIs
raw_data_dir=base_dir/'raw data'
#cleaned data path, will be separated into two main folders which are: market data (contains the data with daily logs) and macroeconomic data
##cleaned market data
cleaned_market_data_dir=base_dir/'cleaned data'/'market_data'
##cleaned macroeconomic data
cleaned_macro_data_dir=base_dir/'cleaned data'/'macroeconomic_data'

In [3]:
#today date 
today = datetime.now().strftime("%d/%m/%Y")

<h4 style="color: #CEDDF4;" >2. Egyptian Market Data</h4>
<h5 style="color: #CEDDF4;" >2.1 CPI Rate from IMF</h5>

In [6]:
#this is the API code for getting the CPI (Consumer Products Index) for Egypt from IMF (International Monetary Fund)
#this API is free

#function to be used at any country with any of published indices of IMF

def imf_data(country,index):
    url=f'https://www.imf.org/external/datamapper/api/v1/{index}/{country}'
    response = requests.get(url)
    data=response.json()
    output=data['values'][index][country]
    df = pd.DataFrame(list(output.items()), columns=['year', index])
    return df

#function to be used for IMF data cleaning 
#cl --> cleaned

def cl_imf_data(dataframe, index, country, start_period, end_period):
    dataframe = dataframe[(dataframe['year'] >= start_period) & (dataframe['year'] <= end_period)].copy()
    dataframe['date'] = pd.to_datetime(dataframe['year'].astype(str) + '-01-01')
    dataframe = dataframe.drop(columns=['year'])
    dataframe = dataframe.rename(columns={index: 'value'})  # ← rename by actual column name, not position
    dataframe = dataframe.set_index('date').sort_index()
    today = pd.Timestamp.today().normalize()
    full_dates = pd.date_range(start='2016-01-01', end=today, freq='D')  # ← start from 01-01, not 01-02
    dataframe = dataframe.reindex(full_dates)
    dataframe = dataframe.ffill()
    dataframe = dataframe.reset_index().rename(columns={'index': 'date'})
    dataframe.insert(loc=0, column='id', value=range(1, len(dataframe) + 1))
    dataframe.insert(loc=1, column='region', value=country)
    dataframe.insert(loc=2, column='ticker', value=index)
    return dataframe

In [7]:
#get CPI index data for egypt
egy_cpi_df = imf_data('EGY','PCPIPCH')
egy_cpi_df.to_csv(raw_data_dir/"egy_cpi.csv", index=False)

#cleaning of CPI index data for egypt
cl_egy_cpi_df=cl_imf_data(egy_cpi_df,'cpi_imf','egypt','2015','2025')
cl_egy_cpi_df.to_csv(cleaned_macro_data_dir/"egy_cpi.csv", index=False)

<h5 style="color: #CEDDF4;" >2.2 Interest Rate from CBE</h5>

In [8]:
#get egyptian data from cbe (central bank of egypt)

#the following options are countermeasure for detecting that we're using code to navigate the website
options=Options()
#options.add_argument("--headless") #To make the code operates in the background
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches",["enable-automation"])
options.add_experimental_option("useAutomationExtension",False)
#option to save the downloaded files into data folders 
prefs= {
    "download.default_directory": str(raw_data_dir),
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=options)

#function to add the needed data for cbe statistics
## will not work on exchange rates till now
def cbe_data(target_data,file_name,start_date):
    cbe_url=f'https://www.cbe.org.eg/en/economic-research/statistics/{target_data.replace(" ","-")}/historical-data'
    driver.get(cbe_url)
    #start_date = "01/01/2016"
    driver.execute_script(f"document.getElementById('fromDate').value = '{start_date}';")
    #end_date="31/12/2025"
    driver.execute_script(f"document.getElementById('toDate').value = '{today}';")
    driver.find_element(By.XPATH,'/html/body/div[1]/section[3]/div[1]/form/div[1]/div/button/div/span[2]').click()
    time.sleep(1)
    driver.find_element(By.XPATH,'//*[@id="historicalDataForm"]/div[2]/button[1]/span').click()
    downloaded_filename = max(raw_data_dir.glob('*'), key=os.path.getmtime)
    downloaded_filename.rename(raw_data_dir / f"{file_name}.xlsx")
    return file_name

In [12]:
cbe_inflation=cbe_data('inflation rates','egy_cbe_inflation_rate','01/01/2016')

In [13]:
def cl_cbe_data(df,country):
    df = pd.read_excel(raw_data_dir/'egy_cbe_inflation_rate.xlsx')
    df=df.drop(columns=['Unnamed: 3','Unnamed: 4'])
    df=df.rename(columns={df.columns[0]:'date',df.columns[1]:'headline_inflation_yy',df.columns[2]:'core_inflation_yy'})
    df['date'] = pd.to_datetime(df['date'], format='%b %Y', errors='coerce').dt.strftime('%d/%m/%Y') #this was made to convert the raw date (feb. 2026 to 01/02/2026)
    df = df.dropna(subset=['date'])
    df = df.reset_index(drop=True)
    #to transpose the values of the multiple tickers
    #column_values=[col for col in df.columns if col != 'date']
    #df=df.melt(id_vars=['date'],value_vars=column_values,var_name='ticker',value_name='value')
    #df['value'] = df['value'].str.replace('%', '').astype(float)
    df.insert(loc=1,column='region',value=country)
    df = df.sort_values(by='date', key=lambda x: pd.to_datetime(x, format='%d/%m/%Y')).reset_index(drop=True) #this was made to sort the dates from oldest to newest
    df.insert(loc=0, column='id', value=range(1, len(df) + 1))
    return df

In [14]:
cl_cbe_data_df=cl_cbe_data(cbe_inflation,'egypt')
cl_cbe_data_df.to_csv(cleaned_macro_data_dir/"egy_cbe_inflation_rates.csv", index=False)

<h5 style="color: #CEDDF4;" >2.3 USD/EGP Exchange Rate from CBE</h5>

In [15]:
#the following options are countermeasure for detecting that we're using code to navigate the website
options=Options()
#options.add_argument("--headless") #To make the code operates in the background
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches",["enable-automation"])
options.add_experimental_option("useAutomationExtension",False)
#option to save the downloaded files into data folders 
prefs= {
    "download.default_directory": str(raw_data_dir),
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=options)

#get exchange rate between usd and egp from cbe
def cbe_usdegp_er(start_date,file_name):
    #file_name='egy_usdegp_rate'
    er_cbe_url='https://www.cbe.org.eg/en/markets/foreign-exchange/foreign-exchange-historical-data'
    driver.get(er_cbe_url)
    time.sleep(5)
    #start_date = "01/01/2016"
    driver.execute_script(f"document.getElementById('fromDate').value = '{start_date}';")
    #end_date="31/12/2025"
    driver.execute_script(f"document.getElementById('toDate').value = '{today}';")
    driver.find_element(By.XPATH,'/html/body/div[1]/section[3]/div[1]/form/div[1]/div/button/div/span[2]/span').click()
    time.sleep(1)
    driver.find_element(By.XPATH,'//*[@id="historicalDataForm"]/div[2]/button[1]').click()
    time.sleep(2)
    downloaded_filename = max(raw_data_dir.glob('*'), key=os.path.getmtime)
    time.sleep(2)
    downloaded_filename.rename(raw_data_dir /f"{file_name}{downloaded_filename.suffix}")
    return file_name


In [16]:
cbe_usdegp_er('01/01/2016','egy_cbe_egpusd_rate')

'egy_cbe_egpusd_rate'

In [20]:
def cl_cbe_usdegp(df, country, ticker):
    df = pd.read_excel(raw_data_dir / 'egy_cbe_egpusd_rate.xlsx')
    df = df.rename(columns={df.columns[0]: 'date', df.columns[1]: 'value'})
    df.insert(loc=0, column='region', value=country)
    df.insert(loc=1, column='ticker', value=ticker)
    df['date'] = pd.to_datetime(df['date'], format='mixed', dayfirst=True, errors='coerce') #convert the date in the excel file from cbe website
    df = df.dropna(subset=['date'])
    df = df.sort_values('date').reset_index(drop=True)
    df = df.set_index('date')
    df = df[~df.index.duplicated(keep='first')]  # remove duplicate dates based on some errors appeared in the run trials
    full_dates = pd.date_range(start='2016-01-01', end=pd.Timestamp.today().normalize(), freq='D')
    df = df.reindex(full_dates)
    df['value'] = df['value'].ffill()
    df = df.reset_index().rename(columns={'index': 'date'})
    df['date'] = df['date'].dt.strftime('%d/%m/%Y')
    df.insert(loc=0, column='id', value=range(1, len(df) + 1))
    return df

In [21]:
cl_cbe_usdegp_rate = cl_cbe_usdegp('usdegp_rate','egypt','egpusd')
cl_cbe_usdegp_rate.to_csv(cleaned_market_data_dir/"egy_egpusd_rate.csv", index=False)

<h4 style="color: #CEDDF4;" >3. Yahoo Finance Data</h4>

In [24]:
#retrieve the global stock and commodity data from yfinance
def yf_data(region,table_ticker,target_data,y_ticker,startdate):
    today_yf = pd.Timestamp.today().strftime('%Y-%m-%d') #today in yahoo finance is different from global today variable because of sorting YMD
    df=yf.download(f"{y_ticker}",start=f'{startdate}',end=today_yf)

    if isinstance(df.columns,pd.MultiIndex):
        df.columns = df.columns.droplevel(1)

    df = df.reset_index()
    df.to_csv(raw_data_dir/f"global_{target_data}_prices.csv", index=False)
    #cleaning of retrieved data from yfinance
    cl_df=df.copy()
    cl_df=cl_df.drop('Volume',axis=1,errors='ignore')
    #cl_df = cl_df.drop(index=0)  # drop second row (index 0 after read_csv)
    cl_df = cl_df.reset_index(drop=True)
    cl_df =cl_df.rename(columns={'Date':'date','Close':'price_usd','High':'high_usd','Low':'low_usd','Open':'open_usd'})
    cl_df['date'] = pd.to_datetime(cl_df['date'])
    cl_df = cl_df.set_index('date').sort_index()
    full_dates = pd.date_range(start=f'{startdate}',end=today_yf, freq='D')# create full date range
    cl_df = cl_df.reindex(full_dates)
    first_valid = cl_df.first_valid_index()
    if first_valid is not None:
        cl_df.loc[:first_valid] = cl_df.loc[:first_valid].bfill()
    cl_df = cl_df.ffill()
    cl_df = cl_df.reset_index().rename(columns={'index':'date'})
    #to transpose the values of the multiple tickers
    #column_values = [col for col in cl_df.columns if col != 'date']
    #cl_df = cl_df.melt(id_vars=['date'], value_vars=column_values, var_name='ticker', value_name='value_in_usd')
    cl_df.insert(loc=0, column='ticker', value=f'{table_ticker}')
    cl_df.insert(loc=0, column='region', value=f'{region}')
    cl_df = cl_df.sort_values('date').reset_index(drop=True)
    cl_df.insert(loc=0, column='id', value=range(1, len(cl_df) + 1))
    cl_df.to_csv(cleaned_market_data_dir/f"{region}_{target_data}_prices.csv", index=False)
    return 

<h4 style="color: #CEDDF4;" >4 Gold Data</h5>
<h5 style="color: #CEDDF4;" >4.1 Global Gold Data</h5>

In [25]:
#yahoo finance ticker is GC=F
yf_data('global','gold','gold','GC=F','2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >4.2 Egyptian Gold Data</h5>

In [27]:
#this data was downloaded manually from https://www.investing.com/currencies/xau-egp-historical-data
egy_gold_df=pd.read_csv(raw_data_dir/ 'egy_gold_prices.csv')
egy_gold_df=egy_gold_df.drop('Vol.',axis=1)
egy_gold_df=egy_gold_df.drop('Change %',axis=1)
egy_gold_df = egy_gold_df.drop(index=0)
egy_gold_df = egy_gold_df.reset_index(drop=True)
egy_gold_df =egy_gold_df.rename(columns={'Date':'date','Price':'price_oz_egp','High':'high_price_egp','Low':'low_price_egp','Open':'open_price_egp'})
egy_gold_df['date'] = pd.to_datetime(egy_gold_df['date'], format='mixed', dayfirst=False)
egy_gold_df = egy_gold_df.set_index('date')
full_dates = pd.date_range(start='01-01-2016',end=today, freq='D')# create full date range
egy_gold_df.index = pd.to_datetime(egy_gold_df.index) 
egy_gold_df = egy_gold_df.reindex(full_dates)
egy_gold_df_first_valid = egy_gold_df.first_valid_index()
if egy_gold_df_first_valid is not None:
    egy_gold_df.loc[:egy_gold_df_first_valid] = egy_gold_df.loc[:egy_gold_df_first_valid].bfill()
egy_gold_df = egy_gold_df.ffill()
egy_gold_df = egy_gold_df.reset_index().rename(columns={'index':'date'})  
#to transpose the values of the multiple tickers
#column_values = [col for col in egy_gold_df.columns if col != 'date']
#egy_gold_df = egy_gold_df.melt(id_vars=['date'], value_vars=column_values, var_name='ticker', value_name='value_in_egp')
egy_gold_df.insert(loc=0, column='ticker', value='gold')
egy_gold_df.insert(loc=0, column='region', value='egypt')
egy_gold_df = egy_gold_df.sort_values('date').reset_index(drop=True)
egy_gold_df.insert(loc=0, column='id', value=range(1, len(egy_gold_df) + 1))
egy_gold_df.to_csv(cleaned_market_data_dir/"egy_gold_prices.csv", index=False)

In [28]:
egy_gold_df

,id,region,ticker,date,price_oz_egp,open_price_egp,high_price_egp,low_price_egp
0,1,egypt,gold,2016-01-01,"8,301.68","8,301.68","8,301.68","8,301.68"
1,2,egypt,gold,2016-01-02,"8,301.68","8,301.68","8,301.68","8,301.68"
2,3,egypt,gold,2016-01-03,"8,301.68","8,301.68","8,301.68","8,301.68"
3,4,egypt,gold,2016-01-04,"8,407.85","8,341.71","8,485.23","8,314.65"
4,5,egypt,gold,2016-01-05,"8,432.60","8,408.26","8,475.37","8,398.91"
...,...,...,...,...,...,...,...,...
3764,3765,egypt,gold,2026-04-22,"265,853.25","268,508.44","272,539.75","264,561.72"
3765,3766,egypt,gold,2026-04-23,"265,853.25","268,508.44","272,539.75","264,561.72"
3766,3767,egypt,gold,2026-04-24,"265,853.25","268,508.44","272,539.75","264,561.72"
3767,3768,egypt,gold,2026-04-25,"265,853.25","268,508.44","272,539.75","264,561.72"


<h4 style="color: #CEDDF4;" >5 Oil Data</h5>
<h5 style="color: #CEDDF4;" >5.1 Brent Oil Data</h5>

In [29]:
#yahoo finance ticker is BZ=F
yf_data('global','brent','brent_oil','BZ=F','2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >5.2 West Texas Intermediate (WTI) Oil Data</h5>

In [30]:
#yahoo finance ticker is CL=F
yf_data('global','wti','wti_price','CL=F','2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >5.3 OPEC Basket Oil Data</h5>

In [34]:
#we will get the data using fred api key which is free and available
#api key is ba5f567693ff05a67920dda63bb2affb
def fred(startdate,series_id,file_name,region,ticker,value,save):
    api_key='ba5f567693ff05a67920dda63bb2affb'
    today_fred = datetime.now().strftime('%Y-%m-%d')  # to change the today from global variable to one that can be used with fred api
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&observation_start={startdate}&observation_end={today_fred}&api_key={api_key}&file_type=json"
    response = requests.get(url)
    data = response.json()
    df = pd.DataFrame(data['observations'])
    df = df.rename(columns={'value': f'{value}'})
    df[f'{value}'] = pd.to_numeric(df[f'{value}'], errors='coerce')
    df['date'] = pd.to_datetime(df['date']) 
    df = df[['date', f'{value}']].set_index('date')
    df.to_csv(raw_data_dir/f"{file_name}.csv", index=False)
    cl_df = df.copy()
    full_dates = pd.date_range(start=startdate, end=today_fred, freq='D')
    cl_df = cl_df.reindex(full_dates)
    first_valid = cl_df.first_valid_index()
    if first_valid is not None:
        cl_df.loc[:first_valid] = cl_df.loc[:first_valid].bfill()
    cl_df[value] = cl_df[value].ffill()
    cl_df = cl_df.reset_index().rename(columns={'index':'date'})
    cl_df.insert(0, 'region', f'{region}')
    cl_df.insert(1, 'ticker', f'{ticker}')
    cl_df.insert(0, 'id', range(1, len(cl_df) + 1))
    cl_df['date'] = cl_df['date'].dt.strftime('%d/%m/%Y')
    save_map={'market': cleaned_market_data_dir,'macro': cleaned_macro_data_dir} #to get the save based on the data we retrieve
    cl_df.to_csv(save_map[save] / f"{file_name}.csv", index=False)
    return


In [35]:
fred('2016-01-01','POILBREUSDM','opec_basket','opec','opec_basket','value','market')

<h5 style="color: #CEDDF4;" >5.4 Oil Volatility Index</h5>

In [36]:
fred('2016-01-01','OVXCLS','oil_vi','global','oil_vi','value','market')

<h5 style="color: #CEDDF4;" >5.5 Gold Volatility Index</h5>

In [37]:
fred('2016-01-01','GVZCLS','gold_vi','global','gold_vi','value','market')

<h4 style="color: #CEDDF4;" >6 Exchange Rate Data</h5>
<h5 style="color: #CEDDF4;" >6.1 Chinese Yuan vs USD</h5>

In [38]:
#yahoo finance ticker is CNY=X
yf_data('global','cnyusd','cnyusd_rate','CNY=X', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >6.2 EURO vs USD</h5>

In [39]:
#yahoo finance ticker is EUR=X
yf_data('global','eurusd','eurusd_rate','EUR=X', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >6.4 Japanese Yen vs USD</h5>

In [40]:
#yahoo finance ticker is JPY=X
yf_data('global','jpyusd','jpyusd_rate','JPY=X', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >6.5 Pound Sterling vs USD</h5>

In [41]:
#yahoo finance ticker is GBP=X
yf_data('global','gbpusd','gbpusd_rate','GBP=X', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >6.6 Norwegian Kroner vs USD</h5>

In [42]:
#yahoo finance ticker is NOK=X
yf_data('global','nokusd','nokusd_rate','NOK=X', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >6.7 Russian Ruble vs USD</h5>

In [43]:
#yahoo finance ticker is RUB=X
yf_data('global','rubusd','rubusd_rate','RUB=X', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >6.8 Swiss Franc vs USD</h5>

In [44]:
#yahoo finance ticker is CHF=X
yf_data('global','chfusd','chfusd_rate','CHF=X', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h4 style="color: #CEDDF4;" >7 Stock Market Data</h5>
<h5 style="color: #CEDDF4;" >7.1 Volatility Index</h5>

In [45]:
#yahoo finance ticker is ^VIX
yf_data('global','vix','vix','^VIX', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >7.2 NASDAQ Composite </h5>

In [46]:
#yahoo finance ticker is ^IXIC
yf_data('usa','nasdaq','nasdaq','^IXIC', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >7.2 Dow Jones Industrial Average </h5>

In [47]:
#yahoo finance ticker is ^DJI
yf_data('usa','dowjones','dowjones','^DJI', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >7.3 S&P 500 </h5>

In [48]:
#yahoo finance ticker is ^GSPC
yf_data('usa','sp500','sp500','^GSPC', '2016-01-01')


[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >7.4 Shanghai Stock Exchange </h5>

In [49]:
#yahoo finance ticker is 000001.SS
yf_data('china','shanghai','shanghai','000001.SS', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >7.5 Japan Stock Exchange </h5>

In [50]:
#yahoo finance ticker is ^N225
yf_data('japan','tokyo','tokyo','^N225', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >7.7 London Stock Exchange </h5>

In [51]:
#yahoo finance ticker is ^EGX30
yf_data('uk','london','london','^FTSE', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >7.8 Hong Kong Stock Exchange </h5>

In [52]:
#yahoo finance ticker is ^HSI
yf_data('hongkong','hongkong','hongkong','^HSI', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >7.9 Egypt Stock Exchange </h5>

In [55]:
#this to get the historical data for egx30 from the official website
#sometime the website is down due to ongoing maintenance, so download the file manually can be the alternative
#the following options are countermeasure for detecting that we're using code to navigate the website
options=Options()
#options.add_argument("--headless") #To make the code operates in the background
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches",["enable-automation"])
options.add_experimental_option("useAutomationExtension",False)
#option to save the downloaded files into data folders 
prefs= {
    "download.default_directory": str(raw_data_dir),
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=options)

egx_url='https://www.egx.com.eg/en/indexdata.aspx?type=1&nav=1'
driver.get(egx_url)
time.sleep(2)
driver.execute_script("document.getElementById('ctl00_C_I_txtDateFrom').value = '01/01/2016';")
time.sleep(2)
driver.execute_script(f"document.getElementById('ctl00_C_I_txtDateTo').value = '{today}';")
time.sleep(5)
driver.execute_script("document.getElementById('ctl00_C_I_btn_Submit').click();")
time.sleep(5)
driver.find_element(By.XPATH,'/html/body/form/table/tbody/tr[2]/td/center/table/tbody/tr[3]/td[2]/table/tbody/tr/td/table/tbody/tr[3]/td/div/div[1]/div/a').click()
downloaded_filename = max(raw_data_dir.glob('*'), key=os.path.getmtime)
downloaded_filename.rename(raw_data_dir / f"egy_egx30_prices{downloaded_filename.suffix}")

JavascriptException: Message: javascript error: Cannot set properties of null (setting 'value')
  (Session info: chrome=147.0.7727.102)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff70df8a8e5+14e45]
	chromedriver!GetHandleVerifier [0x7ff70df8a950+14eb0]
	chromedriver!(No symbol) [0x7ff70dcfd6ed]
	chromedriver!(No symbol) [0x7ff70dd05671]
	chromedriver!(No symbol) [0x7ff70dd08b2f]
	chromedriver!(No symbol) [0x7ff70dda5c1d]
	chromedriver!(No symbol) [0x7ff70dd8002a]
	chromedriver!(No symbol) [0x7ff70dda483b]
	chromedriver!(No symbol) [0x7ff70dd490e8]
	chromedriver!(No symbol) [0x7ff70dd49fc3]
	chromedriver!GetHandleVerifier [0x7ff70e2a0149+32a6a9]
	chromedriver!GetHandleVerifier [0x7ff70e29a715+324c75]
	chromedriver!GetHandleVerifier [0x7ff70e2bc012+346572]
	chromedriver!GetHandleVerifier [0x7ff70dfa7cb5+32215]
	chromedriver!GetHandleVerifier [0x7ff70dfb087c+3addc]
	chromedriver!GetHandleVerifier [0x7ff70df940e4+1e644]
	chromedriver!GetHandleVerifier [0x7ff70df94296+1e7f6]
	chromedri

In [59]:
# cleaning the data for egypt egx30
cl_egx = pd.read_csv(
    raw_data_dir / 'egy_egx30_prices.csv', 
    skiprows=3, 
    names=['date', 'price_egp', 'high_price_egp', 'low_price_egp', 'change_percent'],
    usecols=[0, 1, 2, 3, 4] # Only take the first 5 columns to avoid 'Unnamed' errors
)
cl_egx['date'] = pd.to_datetime(cl_egx['date'], dayfirst=True, errors='coerce')
cl_egx = cl_egx.dropna(subset=['date'])
cl_egx = cl_egx.set_index('date')
full_dates = pd.date_range(start='2016-01-01', end=today, freq='D')
cl_egx = cl_egx.reindex(full_dates)
cl_egx_first_valid = cl_egx.first_valid_index()
if cl_egx_first_valid is not None:
    cl_egx.loc[:cl_egx_first_valid] = cl_egx.loc[:cl_egx_first_valid].bfill()
cl_egx = cl_egx.ffill()
cl_egx = cl_egx.reset_index().rename(columns={'index': 'date'})
cl_egx.insert(0, 'ticker', 'egx30')
cl_egx.insert(0, 'region', 'egypt')
cl_egx.insert(0, 'id', range(1, len(cl_egx) + 1))
cl_egx['date'] = cl_egx['date'].dt.strftime('%d/%m/%Y')
cl_egx.to_csv(cleaned_market_data_dir / "egy_egx30_prices.csv", index=False)

<h4 style="color: #CEDDF4;" >8 Commodities Data</h5>
<h5 style="color: #CEDDF4;" >8.1 Silver</h5>

In [60]:
#yahoo finance ticker is SI=F
yf_data('global','silver','silver','SI=F', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >8.2 Copper</h5>

In [61]:
#yahoo finance ticker is HG=F
yf_data('global','copper','copper','HG=F', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h4 style="color: #CEDDF4;" >9 Geopolitical Risk Indices</h5>
<h5 style="color: #CEDDF4;" >9.1 GPR </h5>

##### This index was made in order to measure of adverse geopolitical events and risks based on a tally of newspaper articles covering geopolitical tensions. it captures "shocks" such as military conflicts, terror threats, and political crises that affect global markets.

In [63]:
#get the raw file for gpr index
gpr_url = 'https://www.matteoiacoviello.com/gpr_files/data_gpr_daily_recent.xls'
gpr_df = pd.read_excel(gpr_url, engine='xlrd') #engine xlrd was used due to compatibility issues with the downloaded files from the website
gpr_df.to_csv(raw_data_dir / "global_gpr_index.csv", index=False)


In [64]:
#cleaning the data for gpr index
gpr_df_cl = gpr_df[['date','GPRD','event']].copy()
gpr_df_cl['date']=pd.to_datetime(gpr_df_cl['date'], format='mixed', dayfirst=False)
gpr_max_date = gpr_df_cl['date'].max()
gpr_df_cl = gpr_df_cl.set_index('date')
gpr_full_dates = pd.date_range(start='01-01-2016',end=gpr_max_date, freq='D')
gpr_df_cl = gpr_df_cl.reindex(gpr_full_dates)
gpr_df_cl['event'] = gpr_df_cl['event'].fillna('no major event')
gpr_df_cl = gpr_df_cl.reset_index().rename(columns={'index':'date'})
gpr_df_cl = gpr_df_cl.sort_values('date').reset_index(drop=True)
gpr_df_cl['date'] = gpr_df_cl['date'].dt.strftime('%#d/%#m/%Y')
gpr_df_cl.columns = ['date', 'gpr', 'event']
gpr_df_cl.insert(loc=0, column='ticker', value='gpr')
gpr_df_cl.insert(loc=0, column='region', value='global')
gpr_df_cl.insert(loc=0, column='id', value=range(1, len(gpr_df_cl) + 1))
gpr_df_cl.to_csv(cleaned_market_data_dir/"global_gpr_index.csv", index=False)

<h5 style="color: #CEDDF4;" >9.2 GPR AI </h5>

In [65]:
ai_gpr_url = 'https://www.matteoiacoviello.com/ai_gpr_files/ai_gpr_data_daily.csv'
ai_gpr_df = pd.read_csv(ai_gpr_url)
ai_gpr_df.to_csv(raw_data_dir / "global_ai_gpr_index.csv", index=False)

In [66]:
#cleaning the data for ai gpr index
ai_gpr_df_cl = ai_gpr_df.copy()
ai_gpr_df_cl.columns = ai_gpr_df_cl.columns.str.lower()
ai_gpr_df_cl['date'] = pd.to_datetime(ai_gpr_df_cl['date'])
ai_gpr_df_cl = ai_gpr_df_cl[ai_gpr_df_cl['date'] >= '2016-01-01']
ai_gpr_max_date = ai_gpr_df_cl['date'].max()
ai_gpr_df_cl = ai_gpr_df_cl.set_index('date')
ai_gpr_full_dates = pd.date_range(start='01-01-2016',end=ai_gpr_max_date, freq='D')
ai_gpr_df_cl = ai_gpr_df_cl.reindex(ai_gpr_full_dates)
ai_gpr_df_cl = ai_gpr_df_cl.reset_index().rename(columns={'index':'date'})
ai_gpr_df_cl = ai_gpr_df_cl.sort_values('date').reset_index(drop=True)
ai_gpr_df_cl.insert(loc=0, column='region', value='global')
ai_gpr_df_cl.insert(loc=0, column='id', value=range(1, len(ai_gpr_df_cl) + 1))

<h4 style="color: #CEDDF4;" >10 Global Economy</h5>
<h5 style="color: #CEDDF4;" >10.1 Euro Zone </h5>

In [67]:
#function to extract the data from european central bank (ecb) api
def ecb_data(ticker,start_date,end_date,df,filename):
    dataflow_id, remainder = ticker.split('.', 1)
    frequency = remainder.split('.')[0] #some data is given in quarterly format
    start_date=pd.to_datetime(start_date, dayfirst=True).strftime('%Y-%m-%d')
    end_date=pd.to_datetime(end_date, dayfirst=True).strftime('%Y-%m-%d')
    ecb_url=f"https://data-api.ecb.europa.eu/service/data/{dataflow_id}/{remainder}?format=csvdata&startPeriod={start_date}&endPeriod={end_date}"
    response = requests.get(ecb_url)
    df = pd.read_csv(io.StringIO(response.text))
    df.to_csv(raw_data_dir/f"{filename}.csv",index=False)
    df_cl=df[['TIME_PERIOD','OBS_VALUE']]
    df_cl=df_cl.rename(columns={'TIME_PERIOD':'date','OBS_VALUE':'value'})
    if frequency =='Q':
        df_cl['date'] = df_cl['date'].str.replace('-Q', 'Q').apply(lambda x: pd.Period(x, freq='Q').to_timestamp())
    else:
        df_cl['date'] = pd.to_datetime(df_cl['date'])
    df_cl['date'] = pd.to_datetime(df_cl['date'])
    df_cl = df_cl.sort_values('date').reset_index(drop=True)
    df_cl['date'] = df_cl['date'].dt.strftime('%#d/%#m/%Y')
    df_cl.insert(loc=0, column='ticker', value='interest_rate')
    df_cl.insert(loc=0, column='region', value='euro_zone')
    df_cl.insert(loc=0, column='id', value=range(1, len(df_cl) + 1))
    df_cl.to_csv(cleaned_macro_data_dir/f"{filename}.csv", index=False)
    return

<h5 style="color: #CEDDF4;" >10.1.1 Interest Rate </h5>

In [68]:
ecb_data('FM.D.U2.EUR.4F.KR.MRR_RT.LEV','01/01/2016','today','ecb_ir','euro_interest_rate')

<h5 style="color: #CEDDF4;" >10.1.2 Inflation Rate </h5>

In [69]:
ecb_data('HICP.M.DE.N.000000.4D0.ANR','01/01/2016','today','ecb_cpi','euro_inflation_rate')

<h5 style="color: #CEDDF4;" >10.1.3 GDP </h5>

In [70]:
ecb_data('MNA.Q.Y.I9.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.N','01/01/2016','today','ecb_cpi','euro_gdp_rate')

In [71]:
ecb_url='https://data-api.ecb.europa.eu/service/data/MNA/Q.Y.I9.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.N?format=csvdata&startPeriod=2016-01-01&endPeriod=2026-03-18'
response = requests.get(ecb_url) #this was done as the main data importing directly will cause lag due to its size
ecb_gdp = pd.read_csv(io.StringIO(response.text))
ecb_gdp.to_csv(raw_data_dir/"euro_gdp_rate.csv",index=False)

<h5 style="color: #CEDDF4;" >10.2 USA </h5>

In [53]:
def fred_data(series_id, startdate, ticker, filename):

    raw_path = raw_data_dir
    raw_path.mkdir(parents=True, exist_ok=True)

    if filename == "dollar_index_historical_data":
        save_clean_path = cleaned_market_data_dir
    else:
        save_clean_path = cleaned_macro_data_dir

    save_clean_path.mkdir(parents=True, exist_ok=True)

    API_KEY = "34e39e6b561a4b741bd157f9c5761267"
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={API_KEY}&file_type=json&observation_start={startdate}"

    response = requests.get(url)
    data = response.json()

    df = pd.DataFrame(data["observations"])

    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df.insert(0, "id", range(1, len(df) + 1))
    df["ticker"] = ticker
    df["region"] = "usa"
    df.dropna(inplace=True)

    df = df[["id", "region", "ticker", "date", "value"]]

    # ===== SAVE RAW =====
    raw_file = raw_path / f"{filename}.csv"
    df.to_csv(raw_file, index=False)

    # ===== SAVE CLEANED =====
    clean_file = save_clean_path / f"{filename}.csv"
    df.to_csv(clean_file, index=False)

    return df

In [54]:

### A dectionary to map FRED series IDs to their corresponding names
fred_mapping = {
    "1Y Treasury Rate Historical Data Monthly": "DGS1",
    "2Y Treasury Rate Historical Data Monthly": "DGS2",
    "5Y Treasury Rate Historical Data Monthly": "DGS5",
    "10Y Treasury Rate Historical Data Monthly": "DGS10",
    "30Y Treasury Rate Historical Data Monthly": "DGS30",
    "Dollar Index Historical Data": "DTWEXBGS",
    "Fed Funds Rate Historical Data Monthly": "FEDFUNDS",
    "Inflation Rate Historical Data Annual Change": "CPIAUCSL",
    "Nominal GDP Historical Data Annual Change": "GDP",
    "Real GDP Historical Data Annual Change": "GDPC1",
}

<h5 style="color: #CEDDF4;" >10.2.1 getting all the data from fred </h5>

In [55]:
for series_name, series_id in fred_mapping.items():
    clean_name = series_name.replace(" ", "_").lower()
    fred_data(series_id, "2016-01-01", clean_name, clean_name)

<h5 style="color: #CEDDF4;" >10.3 Norway </h5>

In [72]:
#function to extract the data from norges bank api
def nor_data(ticker,start_date,end_date,df,filename):
    dataflow_id, remainder = ticker.split('.', 1)
    frequency = remainder.split('.')[0] #some data is given in quarterly format
    start_date=pd.to_datetime(start_date, dayfirst=True).strftime('%Y-%m-%d')
    end_date=pd.to_datetime(end_date, dayfirst=True).strftime('%Y-%m-%d')
    norg_url=f"https://data.norges-bank.no/api/data/{dataflow_id}/{remainder}?format=csv&startPeriod={start_date}&endPeriod={end_date}&locale=en"
    response = requests.get(norg_url)
    df = pd.read_csv(io.StringIO(response.text),sep=';')
    df.to_csv(raw_data_dir/f"{filename}.csv",index=False)
    df_cl=df[['TIME_PERIOD','OBS_VALUE']]
    df_cl=df_cl.rename(columns={'TIME_PERIOD':'date','OBS_VALUE':'value'})
    if frequency =='Q':
        df_cl['date'] = df_cl['date'].str.replace('-Q', 'Q').apply(lambda x: pd.Period(x, freq='Q').to_timestamp())
    else:
        df_cl['date'] = pd.to_datetime(df_cl['date'])
    df_cl['date'] = pd.to_datetime(df_cl['date'])
    df_cl = df_cl.sort_values('date').reset_index(drop=True)
    df_cl['date'] = df_cl['date'].dt.strftime('%#d/%#m/%Y')
    df_cl.insert(loc=0, column='ticker', value='interest_rate')
    df_cl.insert(loc=0, column='region', value='euro_zone')
    df_cl.insert(loc=0, column='id', value=range(1, len(df_cl) + 1))
    df_cl.to_csv(cleaned_macro_data_dir/f"{filename}.csv", index=False)
    return

<h5 style="color: #CEDDF4;" >10.3.1 Interest Rate </h5>

In [73]:
nor_data('IR.B.KPRA.SD.','01/01/2016','today','nor_ir','norway_interest_rate')

In [76]:
def ssb_data(url, query, ticker, region, raw_filename, cleaned_filename, mode='gdp'):
    response = requests.post(url, json=query)
    today_ssb = pd.Timestamp.today().strftime('%Y-%m-%d')

    if mode == 'cpi':
        df = pd.read_csv(io.StringIO(response.text))
        df.to_csv(raw_data_dir / f"{raw_filename}.csv", index=False)
        df_cl = df.transpose().reset_index()
        df_cl = df_cl.drop(index=0).reset_index(drop=True)
        df_cl.columns = ['date', 'value']
        df_cl['date'] = df_cl['date'].str.split().str[-1].str.replace('M', '/')
        df_cl['date'] = pd.to_datetime(df_cl['date'], format='%Y/%m')
        df_cl = df_cl[df_cl['date'] >= '2016-01-01'].reset_index(drop=True)

    elif mode == 'gdp':
        df = pd.read_csv(io.StringIO(response.text), sep=',', quotechar='"')
        df.columns = ['indicator', 'date', 'contents', 'value']
        df['date'] = pd.to_datetime(df['date'].str.replace('M', '-'), format='%Y-%m')
        df = df[['date', 'value']].sort_values('date').reset_index(drop=True)
        df.to_csv(raw_data_dir / f"{raw_filename}.csv", index=False)
        df_cl = df.copy()

    df_cl = df_cl.set_index('date')
    full_dates = pd.date_range(start='2016-01-01', end=today_ssb, freq='D')
    df_cl = df_cl.reindex(full_dates)
    first_valid = df_cl.first_valid_index()
    if first_valid is not None:
        df_cl.loc[:first_valid] = df_cl.loc[:first_valid].bfill()
    df_cl = df_cl.ffill()
    df_cl = df_cl.reset_index().rename(columns={'index': 'date'})
    df_cl['date'] = df_cl['date'].dt.strftime('%d/%m/%Y')
    df_cl.insert(0, 'region', region)
    df_cl.insert(1, 'ticker', ticker)
    df_cl.insert(0, 'id', value=range(1, len(df_cl) + 1))
    df_cl.to_csv(cleaned_macro_data_dir / f"{cleaned_filename}.csv", index=False)
    return

<h5 style="color: #CEDDF4;" >10.3.2 CPI </h5>

In [77]:
ssb_data(
    url="https://data.ssb.no/api/v0/en/table/14710",
    query={"query": [], "response": {"format": "csv"}},
    ticker='cpi', region='norway',
    raw_filename='norway_cpi', cleaned_filename='norway_cpi',
    mode='cpi'
)

<h5 style="color: #CEDDF4;" >10.3.2 GDP </h5>

In [78]:
ssb_data(
    url="https://data.ssb.no/api/v0/en/table/11721/",
    query={"query": [{"code": "Makrost", "selection": {"filter": "item", "values": ["bnpb.nr23_6fn"]}}, {"code": "ContentsCode", "selection": {"filter": "item", "values": ["Priser"]}}, {"code": "Tid", "selection": {"filter": "all", "values": ["*"]}}], "response": {"format": "csv2"}},
    ticker='gdp_mainland', region='norway',
    raw_filename='norway_gdp', cleaned_filename='norway_gdp',
    mode='gdp'
)

<h5 style="color: #CEDDF4;" >10.3.4 10Y Yield </h5>

In [79]:
fred('2016-01-01','IRLTLT01NOM156N','norway_10yy','norway','10y_yield','value','macro')

<h5 style="color: #CEDDF4;" >10.3.5 2Y Yield </h5>

In [80]:
fred('2016-01-01','IR3TIB01NOM156N','norway_2yy','norway','2y_yield','value','macro')

<h5 style="color: #CEDDF4;" >10.4 China </h5>
<h5 style="color: #CEDDF4;" >10.4.1 CPI </h5>

In [81]:
fred('2016-01-01','CHNCPIALLMINMEI','china_cpi','china','cpi','value','macro')

<h5 style="color: #CEDDF4;" >10.4.2 Interest Rate </h5>

In [82]:
fred('2016-01-01','INTDSRCNM193N','china_ir','china','interest_rate','value','macro')

<h5 style="color: #CEDDF4;" >10.4.3 GDP </h5>

In [83]:
fred('2016-01-01','CHNGDPNQDSMEI','china_gdp','china','gdp','value_usd','macro')

<h5 style="color: #CEDDF4;" >10.5 Japan </h5>
<h5 style="color: #CEDDF4;" >10.5.1 CPI </h5>

In [84]:
fred('2016-01-01','JPNCPIALLMINMEI','japan_cpi','japan','cpi','value','macro')

<h5 style="color: #CEDDF4;" >10.5.2 Interest Rate </h5>

In [85]:
fred('2016-01-01','IRLTLT01JPM156N','japan_ir','japan','interest_rate','value','macro')

<h5 style="color: #CEDDF4;" >10.5.3 GDP </h5>

In [86]:
fred('2016-01-01','JPNGDPNQDSMEI','japan_gdp','japan','gdp','value','macro')

<h5 style="color: #CEDDF4;" >10.5.4 10Y Yield </h5>

In [87]:
fred('2016-01-01','IRLTLT01JPM156N','japan_10yy','japan','10y_yield','value','macro')

<h5 style="color: #CEDDF4;" >10.5.5 2Y Yield </h5>

In [88]:
fred('2016-01-01','IR3TIB01JPM156N','japan_2yy','japan','2y_yield','value','macro')

<h4 style="color: #CEDDF4;" >11 Global Energy</h5>

In [6]:
#as opec doesn't have a dedicated api to get the information about energy production/usage, world bank has such data for each countries
#some countries updates is slower that updates for usa - as example - but its a reliable source for data
def wb_energy(countries, indicators, save): #wb stands for world bank
    save_map = {'market': cleaned_market_data_dir, 'macro': cleaned_macro_data_dir}
    for country_code, country_name in countries.items():
        country_dfs = []
        for ticker, indicator_code in indicators.items():
            try:
                url = f"https://api.worldbank.org/v2/country/{country_code}/indicator/{indicator_code}?format=json&date=2016:2025&per_page=100"
                response = requests.get(url)
                data = response.json()
                df = pd.DataFrame(data[1])[['date', 'value']]
                df = df.rename(columns={'value': ticker})
                df[ticker] = pd.to_numeric(df[ticker], errors='coerce')
                df['date'] = pd.to_datetime(df['date'], format='%Y')
                df = df.dropna(subset=[ticker]).sort_values('date').reset_index(drop=True)
                df = df.set_index('date')
                full_dates = pd.date_range(start='2016-01-01', end=pd.Timestamp.today(), freq='D')
                df = df.reindex(full_dates)
                first_valid = df.first_valid_index()
                if first_valid is not None:
                    df.loc[:first_valid] = df.loc[:first_valid].bfill()
                df = df.ffill()
                
                df = df.reset_index().rename(columns={'index': 'date'})
                df.to_csv(raw_data_dir / f"{country_name}_{ticker}.csv", index=False)
                country_dfs.append(df)
            except Exception as e:
                pass

        if country_dfs:
            merged = country_dfs[0]
            for df in country_dfs[1:]:
                merged = pd.merge(merged, df, on='date', how='outer')

            merged = merged.sort_values('date').reset_index(drop=True)
            merged['date'] = merged['date'].dt.strftime('%d/%m/%Y')
            merged.insert(0, 'region', country_name)
            merged.insert(0, 'id', range(1, len(merged) + 1))
            merged.to_csv(save_map[save] / f"{country_name}_energy.csv", index=False)
    return

In [7]:
#world bank tickers
tickers = {
    'oil_elec_pct': 'EG.ELC.PETR.ZS', #this is the electricity production percentage comes from oil
    'energy_use':   'EG.USE.PCAP.KG.OE', #energy use per capita, total energy used per person
    'fuel_imports': 'TM.VAL.FUEL.ZS.UN', #percentage of country's imports which is spent on fuel (all types)
    'fuel_exports': 'TX.VAL.FUEL.ZS.UN', #country's total export for fuel
    'gas_elec_pct': 'EG.ELC.NGAS.ZS', #this is the electricity production percentage comes from gas
}
#needed countries list
countries = {
    'US': 'usa',    'EG': 'egypt',   'DE': 'germany',
    'NO': 'norway', 'CN': 'china',   'JP': 'japan',
    'RU': 'russia', 'SA': 'ksa',     'KW': 'kuwait',
    'QA': 'qatar',  'AE': 'uae'
}

wb_energy(countries, tickers, 'macro')
